# Discovery Engine — Quickstart

This notebook walks through a complete Discovery Engine run: uploading a dataset, waiting for the analysis to finish, and working with the results.

**What Discovery Engine does:** Finds novel, statistically validated patterns in tabular data — feature interactions, subgroup effects, and conditional relationships you would not think to look for. Each pattern is validated on hold-out data with FDR-corrected p-values and checked against academic literature for novelty.

Get an API key at [disco.leap-labs.com/developers](https://disco.leap-labs.com/developers) or sign up below.

In [ ]:
# Install the SDK (run once)
%pip install discovery-engine-api[jupyter] pandas --quiet

## Get an API Key

If you already have a key, skip this cell. Otherwise, sign up with your email — free tier includes unlimited public runs.

In [ ]:
import requests

email = "you@example.com"  # Replace with your email

# Step 1: Request verification code
r = requests.post(
    "https://disco.leap-labs.com/api/signup",
    json={"email": email},
)
print(r.json())
# → {"status": "verification_required", "email": "..."}

In [ ]:
# Step 2: Submit the code from your email
code = "123456"  # Replace with code from email

r = requests.post(
    "https://disco.leap-labs.com/api/signup/verify",
    json={"email": email, "code": code},
)
data = r.json()
api_key = data["key"]
print(f"API key: {api_key}")
print(f"Credits: {data['credits']} | Tier: {data['tier']}")

## Prepare a Dataset

Discovery Engine works with any tabular dataset. You need:
- A **target column** — the outcome you want to understand (e.g. treatment response, churn, yield)
- **Feature columns** — everything that might be related to it

Below we create a synthetic clinical dataset. In practice, pass any CSV, Excel, Parquet, or Pandas DataFrame.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 500

age = rng.integers(25, 75, n)
bmi = rng.normal(27, 5, n).clip(16, 45)
blood_pressure = rng.normal(120, 18, n).clip(80, 180)
cholesterol = rng.normal(200, 40, n).clip(120, 320)
smoking = rng.choice(["never", "former", "current"], n, p=[0.5, 0.3, 0.2])
exercise_days_per_week = rng.integers(0, 8, n)
diet_score = rng.integers(1, 11, n)  # 1–10

# Simulate a target with real structure baked in:
# Older patients with high BP and low exercise respond poorly
treatment_response = (
    50
    - 0.3 * np.maximum(age - 55, 0)
    - 0.2 * np.maximum(blood_pressure - 130, 0)
    + 2 * exercise_days_per_week
    + 1.5 * diet_score
    + rng.normal(0, 8, n)
).clip(0, 100)

df = pd.DataFrame({
    "patient_id": [f"P{i:04d}" for i in range(n)],
    "age": age,
    "bmi": bmi.round(1),
    "blood_pressure": blood_pressure.round(0).astype(int),
    "cholesterol": cholesterol.round(0).astype(int),
    "smoking": smoking,
    "exercise_days_per_week": exercise_days_per_week,
    "diet_score": diet_score,
    "treatment_response": treatment_response.round(1),
})

print(df.shape)
df.head()

## Run Discovery Engine

Pass the DataFrame directly. `engine.run()` submits the job and waits for completion (runs take 3–15 minutes).

Setting `visibility="public"` makes the run free — results are published to the public gallery. Use `visibility="private"` to keep results private (costs credits).

Provide `column_descriptions` for any non-obvious column names — it significantly improves pattern explanations.

In [ ]:
from discovery import Engine

api_key = "disco_..."  # Replace with your key

engine = Engine(api_key=api_key)

result = engine.run(
    file=df,
    target_column="treatment_response",
    title="Clinical trial response dataset",
    description="Synthetic dataset modelling treatment response in a clinical cohort.",
    column_descriptions={
        "bmi": "Body mass index",
        "blood_pressure": "Systolic blood pressure (mmHg)",
        "cholesterol": "Total cholesterol (mg/dL)",
        "exercise_days_per_week": "Number of days per week with ≥30 min exercise",
        "diet_score": "Self-reported diet quality score (1=poor, 10=excellent)",
        "treatment_response": "Treatment response score (0–100, higher is better)",
    },
    excluded_columns=["patient_id"],  # Exclude identifiers
    visibility="public",
    wait=True,
)

print(f"Status: {result.status}")
print(f"Patterns found: {len(result.patterns)}")
print(f"Report: {result.report_url}")

## Explore the Results

Results contain **patterns** — each is a combination of conditions (feature ranges/values) that together define a subgroup with a meaningfully different outcome. Not single correlations.

In [ ]:
# High-level summary
if result.summary:
    print(result.summary.overview)
    print()
    for insight in result.summary.key_insights:
        print(f"  • {insight}")

In [ ]:
# Novel patterns — findings not in the existing literature
novel = [
    p for p in result.patterns
    if p.novelty_type == "novel" and p.p_value < 0.05
]

print(f"{len(novel)} novel patterns (p < 0.05)\n")

for p in novel[:5]:
    print(f"Pattern: {p.description}")
    print(f"  p={p.p_value:.4f} | effect={p.abs_target_change:+.2f} | support={p.support_percentage:.1f}%")
    print(f"  Why novel: {p.novelty_explanation}")
    if p.citations:
        print(f"  Checked against {len(p.citations)} paper(s)")
    print()

In [ ]:
# Inspect the conditions of the top pattern
if novel:
    top = novel[0]
    print(f"Top pattern: {top.description}\n")
    for cond in top.conditions:
        if cond["type"] == "continuous":
            print(f"  {cond['feature']}: {cond['min_value']} — {cond['max_value']}")
        elif cond["type"] == "categorical":
            print(f"  {cond['feature']}: {cond['values']}")

In [ ]:
# Feature importance — computed via Hierarchical Perturbation (HiPe)
# Scores are signed: positive = increases prediction, negative = decreases it
if result.feature_importance:
    scores = sorted(
        result.feature_importance.scores,
        key=lambda s: abs(s.score),
        reverse=True,
    )
    print("Feature importance (signed):")
    for s in scores:
        bar = "█" * int(abs(s.score) * 20)
        direction = "+" if s.score >= 0 else "-"
        print(f"  {s.feature:<30} {direction}{bar}")

In [ ]:
# Interactive report — share this link to explore results visually
print(f"Interactive report: {result.report_url}")

## Next Steps

- **Deeper analysis:** Set `depth_iterations=2` (or higher) with `visibility="private"` to find more patterns
- **Cost estimate:** Use `engine.estimate(file_size_mb=..., num_columns=..., depth_iterations=...)` before a paid run
- **Submit without waiting:** Use `engine.run_async(wait=False)` then `engine.wait_for_completion(run_id)` later
- **Full SDK reference:** https://github.com/leap-laboratories/discovery-engine/blob/main/docs/python-sdk.md
- **MCP server:** https://github.com/leap-laboratories/discovery-engine/blob/main/SKILL.md